In [17]:
# CELL 0: Environment Setup, Warnings Suppression & Hardware Inspection
import os
import json
import random
import warnings
import requests
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score

# Suppress deprecation and runtime warnings globally
warnings.filterwarnings("ignore")

# Set random seeds for strict reproducibility
SEED = 42
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
random.seed(SEED)

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("=== ENVIRONMENT & HARDWARE INSPECTION ===")
print(f"PyTorch Version   : {torch.__version__}")
print(f"CUDA Available    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Target GPU Device : {torch.cuda.get_device_name(0)}")
print(f"Active Device     : {device}")

# Dataset directory resolution
PRIMARY_DATA_DIR = r"D:\GitHub\fedkg-maint\data"
DATA_DIR = PRIMARY_DATA_DIR if os.path.exists(PRIMARY_DATA_DIR) else "./data"
print(f"Resolved DATA_DIR : {os.path.abspath(DATA_DIR)}")

ModuleNotFoundError: No module named 'tqdm'

## CELL 1: Data Ingestion & Defensive Feature Engineering

In [ ]:
print("Loading Microsoft Azure Predictive Maintenance Dataset CSV files...")

telemetry_path = os.path.join(DATA_DIR, "telemetry.csv")
errors_path = os.path.join(DATA_DIR, "errors.csv")
maint_path = os.path.join(DATA_DIR, "maint.csv")
failures_path = os.path.join(DATA_DIR, "failures.csv")
machines_path = os.path.join(DATA_DIR, "machines.csv")

# Load raw CSV files directly
df_telemetry = pd.read_csv(telemetry_path)
df_errors = pd.read_csv(errors_path)
df_maint = pd.read_csv(maint_path)
df_failures = pd.read_csv(failures_path)
df_machines = pd.read_csv(machines_path)

# Parse datetimes
df_telemetry['datetime'] = pd.to_datetime(df_telemetry['datetime'])
df_errors['datetime'] = pd.to_datetime(df_errors['datetime'])
df_maint['datetime'] = pd.to_datetime(df_maint['datetime'])
df_failures['datetime'] = pd.to_datetime(df_failures['datetime'])

# Construct hourly backbone per machine
df_telemetry = df_telemetry.sort_values(['machineID', 'datetime']).reset_index(drop=True)

# 1. Rolling 24h mean & std for telemetry
telemetry_cols = ['volt', 'rotate', 'pressure', 'vibration']
for col in telemetry_cols:
    df_telemetry[f'{col}_mean_24h'] = df_telemetry.groupby('machineID')[col].transform(
        lambda x: x.rolling(24, min_periods=24).mean()
    )
    df_telemetry[f'{col}_std_24h'] = df_telemetry.groupby('machineID')[col].transform(
        lambda x: x.rolling(24, min_periods=24).std()
    )

# 2. Logged error counts in 24h window
df_errors['error_flag'] = 1
error_dummies = pd.pivot_table(
    df_errors, index=['datetime', 'machineID'], columns='errorID', values='error_flag', aggfunc='sum', fill_value=0
).reset_index()

for err in ['error1', 'error2', 'error3', 'error4', 'error5']:
    if err not in error_dummies.columns:
        error_dummies[err] = 0

feature_df = pd.merge(df_telemetry, error_dummies, on=['datetime', 'machineID'], how='left').fillna(0)

for err in ['error1', 'error2', 'error3', 'error4', 'error5']:
    feature_df[f'{err}_count_24h'] = feature_df.groupby('machineID')[err].transform(
        lambda x: x.rolling(24, min_periods=1).sum()
    )

# 3. Cumulative days elapsed since last component maintenance replacement
maint_dummies = pd.pivot_table(
    df_maint, index=['datetime', 'machineID'], columns='comp', values='machineID', aggfunc='count', fill_value=0
).reset_index()

for comp in ['comp1', 'comp2', 'comp3', 'comp4']:
    if comp not in maint_dummies.columns:
        maint_dummies[comp] = 0

temp_maint_cols = []
for comp in ['comp1', 'comp2', 'comp3', 'comp4']:
    m_sub = df_maint[df_maint['comp'] == comp][['datetime', 'machineID']].copy()
    col_name = f'last_maint_{comp}'
    temp_maint_cols.append(col_name)
    m_sub[col_name] = m_sub['datetime']
    
    feature_df = pd.merge(feature_df, m_sub[['datetime', 'machineID', col_name]], on=['datetime', 'machineID'], how='left')
    feature_df[col_name] = feature_df.groupby('machineID')[col_name].ffill()
    
    days_elapsed = (feature_df['datetime'] - feature_df[col_name]).dt.total_seconds() / (24 * 3600)
    feature_df[f'days_since_{comp}'] = days_elapsed.fillna(365.0)

# Clean drop without inplace warning
feature_df = feature_df.drop(columns=temp_maint_cols)

# 4. Construct binary target: failure within next 24 hours
df_failures['failure_flag'] = 1
fail_summary = df_failures.groupby(['datetime', 'machineID'])['failure_flag'].max().reset_index()

feature_df = pd.merge(feature_df, fail_summary, on=['datetime', 'machineID'], how='left')
feature_df['failure_flag'] = feature_df['failure_flag'].fillna(0)

feature_df['failure_next_24h'] = feature_df.groupby('machineID')['failure_flag'].transform(
    lambda x: x.iloc[::-1].rolling(24, min_periods=1).max().iloc[::-1]
).astype(int)

# Final cleanup
feature_df = feature_df.drop(columns=['error1', 'error2', 'error3', 'error4', 'error5', 'failure_flag'])
feature_df = feature_df.dropna().reset_index(drop=True)

print(f"Feature engineering complete. Total processed rows: {len(feature_df):,}")
print(f"Target distribution (failure_next_24h):\n{feature_df['failure_next_24h'].value_counts()}")

## CELL 2: Non-IID Factory Splitting & Scaling with Validation Sets

In [ ]:
merged_df = pd.merge(feature_df, df_machines[['machineID', 'model']], on='machineID', how='left')

feature_columns = [
    'volt', 'rotate', 'pressure', 'vibration',
    'volt_mean_24h', 'volt_std_24h', 'rotate_mean_24h', 'rotate_std_24h',
    'pressure_mean_24h', 'pressure_std_24h', 'vibration_mean_24h', 'vibration_std_24h',
    'error1_count_24h', 'error2_count_24h', 'error3_count_24h', 'error4_count_24h', 'error5_count_24h',
    'days_since_comp1', 'days_since_comp2', 'days_since_comp3', 'days_since_comp4'
]

factory_assignments = {
    'Factory 1 (Client 1)': 'model1',
    'Factory 2 (Client 2)': 'model2',
    'Factory 3 (Client 3)': 'model3',
    'Factory 4 (Held-Out Test)': 'model4'
}

client_data = {}
summary_rows = []

for factory_name, model_type in factory_assignments.items():
    sub_df = merged_df[merged_df['model'] == model_type].copy().reset_index(drop=True)
    
    X_raw = sub_df[feature_columns].values
    y_raw = sub_df['failure_next_24h'].values
    
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_raw)
    
    pos_count = int(np.sum(y_raw == 1))
    neg_count = int(np.sum(y_raw == 0))
    pos_weight_val = float(neg_count / max(pos_count, 1))
    
    X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
    y_tensor = torch.tensor(y_raw, dtype=torch.float32).unsqueeze(1)
    
    if factory_name != 'Factory 4 (Held-Out Test)':
        # 80/20 Train/Validation Split for Early Stopping
        val_size = int(len(sub_df) * 0.2)
        train_size = len(sub_df) - val_size
        
        X_train, X_val = X_tensor[:train_size], X_tensor[train_size:]
        y_train, y_val = y_tensor[:train_size], y_tensor[train_size:]
        
        train_ds = TensorDataset(X_train, y_train)
        val_ds = TensorDataset(X_val, y_val)
        
        client_data[factory_name] = {
            'df': sub_df,
            'dataset': train_ds,
            'val_dataset': val_ds,
            'loader': DataLoader(train_ds, batch_size=256, shuffle=True),
            'val_loader': DataLoader(val_ds, batch_size=256, shuffle=False),
            'X': X_tensor, 'y': y_tensor,
            'pos_weight': pos_weight_val
        }
    else:
        test_ds = TensorDataset(X_tensor, y_tensor)
        client_data[factory_name] = {
            'df': sub_df,
            'dataset': test_ds,
            'loader': DataLoader(test_ds, batch_size=256, shuffle=False),
            'X': X_tensor, 'y': y_tensor,
            'pos_weight': pos_weight_val
        }
        
    summary_rows.append({
        'Factory Name': factory_name,
        'Machine Model': model_type,
        'Row Count': len(sub_df),
        'Positives': pos_count,
        'Pos Ratio (%)': f"{(pos_count / len(sub_df)) * 100:.2f}%",
        'Pos Weight (BCE)': round(pos_weight_val, 2)
    })

df_factory_summary = pd.DataFrame(summary_rows)
print("\n=== FACTORY CLIENT DATASET SUMMARY ===")
print(df_factory_summary.to_string(index=False))

## CELL 3: Model Architectures (MLP, 1D-CNN, LSTM, TCN, Autoencoder)

In [ ]:
# 1. Multi-Layer Perceptron (Dense Tabular)
class MLPClassifier(nn.Module):
    def __init__(self, input_dim):
        super(MLPClassifier, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1)
        )
    def forward(self, x):
        return self.net(x)

# 2. 1D Convolutional Neural Network
class CNN1DClassifier(nn.Module):
    def __init__(self, input_dim):
        super(CNN1DClassifier, self).__init__()
        self.conv1 = nn.Conv1d(in_channels=input_dim, out_channels=32, kernel_size=1)
        self.relu1 = nn.ReLU()
        self.conv2 = nn.Conv1d(in_channels=32, out_channels=16, kernel_size=1)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(16, 1)
    def forward(self, x):
        x = x.unsqueeze(2)
        x = self.conv1(x)
        x = self.relu1(x)
        x = self.conv2(x)
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

# 3. Long Short-Term Memory (Recurrent)
class LSTMClassifier(nn.Module):
    def __init__(self, input_dim):
        super(LSTMClassifier, self).__init__()
        self.lstm = nn.LSTM(input_size=input_dim, hidden_size=32, num_layers=2, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(32, 1)
    def forward(self, x):
        x = x.unsqueeze(1) # Shape: (Batch, Seq_Len=1, Features)
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

# 4. Temporal Convolutional Network (TCN)
class TCNClassifier(nn.Module):
    def __init__(self, input_dim):
        super(TCNClassifier, self).__init__()
        # Causal Dilated Conv Block
        self.conv1 = nn.Conv1d(1, 16, kernel_size=3, padding=2, dilation=2)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(0.2)
        self.conv2 = nn.Conv1d(16, 32, kernel_size=3, padding=4, dilation=4)
        self.relu2 = nn.ReLU()
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(32, 1)
    def forward(self, x):
        x = x.unsqueeze(1) # Shape: (Batch, Channels=1, Features)
        x = self.dropout1(self.relu1(self.conv1(x)))
        x = self.relu2(self.conv2(x))
        x = self.pool(x).squeeze(2)
        return self.fc(x)

# 5. Unsupervised Autoencoder Anomaly Classifier
class AutoencoderClassifier(nn.Module):
    def __init__(self, input_dim):
        super(AutoencoderClassifier, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 8)
        )
        self.decoder = nn.Sequential(
            nn.Linear(8, 16),
            nn.ReLU(),
            nn.Linear(16, input_dim)
        )
        # Classification head accepting Latent (8) + Reconstruction Loss Feature (1) = 9
        self.classifier = nn.Sequential(
            nn.Linear(9, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )
    def forward(self, x):
        latent = self.encoder(x)
        reconstructed = self.decoder(latent)
        recon_error = torch.mean((x - reconstructed) ** 2, dim=1, keepdim=True)
        combined = torch.cat([latent, recon_error], dim=1)
        return self.classifier(combined)

input_feature_dim = len(feature_columns)
print(f"Initialized 5 Model Architectures for Input Dimension: {input_feature_dim}")

## CELL 4: Early Stopping Helper & Baseline Models Training

In [ ]:
class EarlyStopping:
    def __init__(self, patience=3, min_delta=1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = float('inf')
        self.counter = 0
        self.early_stop = False
        self.best_state_dict = None

    def __call__(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            self.best_state_dict = {k: v.cpu().clone().detach() for k, v in model.state_dict().items()}
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True

def train_model(model, train_loader, val_loader=None, epochs=15, lr=1e-3, pos_weight_val=1.0, patience=3):
    model = model.to(device)
    pos_weight_tensor = torch.tensor([pos_weight_val], dtype=torch.float32).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    early_stopping = EarlyStopping(patience=patience)
    epoch_losses = []

    pbar = tqdm(range(epochs), desc=f"Training {model.__class__.__name__}", leave=False)
    for epoch in pbar:
        model.train()
        running_loss = 0.0
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad()
            logits = model(batch_x)
            loss = criterion(logits, batch_y)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * batch_x.size(0)
            
        train_loss = running_loss / len(train_loader.dataset)
        epoch_losses.append(train_loss)
        
        # Evaluate validation loss for early stopping
        if val_loader is not None:
            model.eval()
            val_loss_sum = 0.0
            with torch.no_grad():
                for vx, vy in val_loader:
                    vx, vy = vx.to(device), vy.to(device)
                    v_logits = model(vx)
                    v_loss = criterion(v_logits, vy)
                    val_loss_sum += v_loss.item() * vx.size(0)
            val_loss = val_loss_sum / len(val_loader.dataset)
            
            pbar.set_postfix({'Train Loss': f"{train_loss:.4f}", 'Val Loss': f"{val_loss:.4f}"})
            early_stopping(val_loss, model)
            if early_stopping.early_stop:
                model.load_state_dict({k: v.to(device) for k, v in early_stopping.best_state_dict.items()})
                break
        else:
            pbar.set_postfix({'Train Loss': f"{train_loss:.4f}"})
            
    return model, epoch_losses

# 1. Centralized Baseline Training
print("=== TRAINING CENTRALIZED BASELINE MODELS ===")
cent_X = torch.cat([client_data[c]['X'] for c in ['Factory 1 (Client 1)', 'Factory 2 (Client 2)', 'Factory 3 (Client 3)']], dim=0)
cent_y = torch.cat([client_data[c]['y'] for c in ['Factory 1 (Client 1)', 'Factory 2 (Client 2)', 'Factory 3 (Client 3)']], dim=0)

cent_ds = TensorDataset(cent_X, cent_y)
cent_loader = DataLoader(cent_ds, batch_size=256, shuffle=True)
cent_pw = (len(cent_y) - cent_y.sum().item()) / max(cent_y.sum().item(), 1)

architectures = {
    'MLP': MLPClassifier,
    '1D-CNN': CNN1DClassifier,
    'LSTM': LSTMClassifier,
    'TCN': TCNClassifier,
    'Autoencoder': AutoencoderClassifier
}

centralized_models = {}
for name, arch_cls in tqdm(architectures.items(), desc="Centralized Architectures"):
    m, losses = train_model(arch_cls(input_feature_dim), cent_loader, epochs=15, lr=1e-3, pos_weight_val=cent_pw)
    centralized_models[name] = m
    print(f"Centralized {name:11s} | Final Loss: {losses[-1]:.4f}")

# 2. Local Baselines Training
print("\n=== TRAINING LOCAL BASELINE MODELS ===")
local_models = {}
for factory in ['Factory 1 (Client 1)', 'Factory 2 (Client 2)', 'Factory 3 (Client 3)']:
    t_loader = client_data[factory]['loader']
    v_loader = client_data[factory]['val_loader']
    pw = client_data[factory]['pos_weight']
    
    for name, arch_cls in architectures.items():
        m, losses = train_model(arch_cls(input_feature_dim), t_loader, val_loader=v_loader, epochs=15, lr=1e-3, pos_weight_val=pw)
        local_models[f"{factory} - {name}"] = m
        print(f"{factory} -> {name:11s} | Final Loss: {losses[-1]:.4f}")

## CELL 5: Federated Learning Routines (FedAvg, FedProx, FedAvg+DP)

In [ ]:
def copy_model_params(model):
    return {k: v.clone().detach() for k, v in model.state_dict().items()}

def run_federated_training(model_class, clients, mode="FedAvg", rounds=10, local_epochs=3, lr=1e-3, mu=0.01, clip_C=1.0, sigma=0.01, patience=3):
    global_model = model_class(input_feature_dim).to(device)
    client_names = list(clients.keys())
    total_samples = sum([len(clients[c]['dataset']) for c in client_names])
    client_weights = {c: len(clients[c]['dataset']) / total_samples for c in client_names}
    
    early_stopping = EarlyStopping(patience=patience)
    round_losses = []
    
    r_pbar = tqdm(range(rounds), desc=f"FL [{mode} - {model_class.__name__}]")
    for r in r_pbar:
        global_weights = copy_model_params(global_model)
        local_weight_updates = []
        round_loss_sum = 0.0
        
        for c_name in client_names:
            client_loader = clients[c_name]['loader']
            pw_val = clients[c_name]['pos_weight']
            
            local_model = model_class(input_feature_dim).to(device)
            local_model.load_state_dict(global_weights)
            local_model.train()
            
            pos_weight_tensor = torch.tensor([pw_val], dtype=torch.float32).to(device)
            criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
            optimizer = optim.Adam(local_model.parameters(), lr=lr)
            
            for ep in range(local_epochs):
                for batch_x, batch_y in client_loader:
                    batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                    optimizer.zero_grad()
                    logits = local_model(batch_x)
                    loss = criterion(logits, batch_y)
                    
                    if mode == "FedProx":
                        prox_term = 0.0
                        for n_param, param in local_model.named_parameters():
                            prox_term += (torch.norm(param - global_weights[n_param].to(device)) ** 2)
                        loss += (mu / 2.0) * prox_term
                        
                    loss.backward()
                    optimizer.step()
                    round_loss_sum += loss.item() * batch_x.size(0)
            
            local_weight_updates.append((client_weights[c_name], copy_model_params(local_model)))
            
        avg_round_loss = round_loss_sum / (total_samples * local_epochs)
        round_losses.append(avg_round_loss)
        
        # Parameter Aggregation Phase
        new_global_state = {}
        for key in global_weights.keys():
            if mode in ["FedAvg", "FedProx"]:
                stacked = torch.stack([w * local_state[key] for w, local_state in local_weight_updates], dim=0)
                new_global_state[key] = torch.sum(stacked, dim=0)
            elif mode == "FedAvg_DP":
                # Safe check for integer buffers (skips BatchNorm tracking counters)
                if not torch.is_floating_point(global_weights[key]):
                    new_global_state[key] = global_weights[key]
                    continue
                
                deltas = []
                for w, local_state in local_weight_updates:
                    delta = (local_state[key] - global_weights[key]).float()
                    delta_norm = torch.norm(delta)
                    clip_factor = max(1.0, (delta_norm / clip_C).item())
                    clipped_delta = delta / clip_factor
                    deltas.append(w * clipped_delta)
                
                summed_delta = torch.sum(torch.stack(deltas, dim=0), dim=0)
                noise = torch.randn_like(summed_delta) * sigma
                new_global_state[key] = global_weights[key] + summed_delta + noise
                
        global_model.load_state_dict(new_global_state)
        
        # Global Validation Early Stopping Check
        val_loss_sum = 0.0
        total_val_samples = 0
        global_model.eval()
        with torch.no_grad():
            for c_name in client_names:
                v_loader = clients[c_name]['val_loader']
                pw_val = clients[c_name]['pos_weight']
                pw_tensor = torch.tensor([pw_val], dtype=torch.float32).to(device)
                crit = nn.BCEWithLogitsLoss(pos_weight=pw_tensor)
                for vx, vy in v_loader:
                    vx, vy = vx.to(device), vy.to(device)
                    v_loss = crit(global_model(vx), vy)
                    val_loss_sum += v_loss.item() * vx.size(0)
                    total_val_samples += vx.size(0)
        
        glob_val_loss = val_loss_sum / max(total_val_samples, 1)
        r_pbar.set_postfix({'Train Loss': f"{avg_round_loss:.4f}", 'Val Loss': f"{glob_val_loss:.4f}"})
        
        early_stopping(glob_val_loss, global_model)
        if early_stopping.early_stop:
            global_model.load_state_dict({k: v.to(device) for k, v in early_stopping.best_state_dict.items()})
            print(f"[{mode} - {model_class.__name__}] Early stopping triggered at Round {r+1}")
            break
            
    return global_model, round_losses

fl_clients = {k: client_data[k] for k in ['Factory 1 (Client 1)', 'Factory 2 (Client 2)', 'Factory 3 (Client 3)']}

print("\n=== RUNNING FEDERATED LEARNING PIPELINES ACROSS ALL ARCHITECTURES ===")
fl_models = {}
fl_histories = {}

for arch_name, arch_cls in architectures.items():
    print(f"\n--- Running FL Pipelines for Architecture: {arch_name} ---")
    fedavg_m, fedavg_h = run_federated_training(arch_cls, fl_clients, mode="FedAvg", rounds=10)
    fedprox_m, fedprox_h = run_federated_training(arch_cls, fl_clients, mode="FedProx", rounds=10, mu=0.01)
    feddp_m, feddp_h = run_federated_training(arch_cls, fl_clients, mode="FedAvg_DP", rounds=10, clip_C=1.0, sigma=0.01)
    
    fl_models[f"FedAvg ({arch_name})"] = fedavg_m
    fl_models[f"FedProx ({arch_name})"] = fedprox_m
    fl_models[f"FedAvg+DP ({arch_name})"] = feddp_m
    
    fl_histories[f"FedAvg ({arch_name})"] = fedavg_h
    fl_histories[f"FedProx ({arch_name})"] = fedprox_h
    fl_histories[f"FedAvg+DP ({arch_name})"] = feddp_h

## CELL 6: Cross-Site Evaluation on Held-Out Factory 4

In [ ]:

def evaluate_model(model, loader, device=device):
    model.eval()
    y_true = []
    y_scores = []
    with torch.no_grad():
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(device)
            logits = model(batch_x)
            probs = torch.sigmoid(logits).cpu().numpy().flatten()
            y_scores.extend(probs)
            y_true.extend(batch_y.numpy().flatten())
            
    y_true = np.array(y_true)
    y_scores = np.array(y_scores)
    y_pred = (y_scores >= 0.5).astype(int)
    
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    pr_auc = average_precision_score(y_true, y_scores)
    
    return prec, rec, f1, pr_auc, y_scores

test_loader = client_data['Factory 4 (Held-Out Test)']['loader']

all_eval_models = {}
# Add Centralized Models
for name, m in centralized_models.items():
    all_eval_models[f"Centralized ({name})"] = m
# Add FL Models
all_eval_models.update(fl_models)

eval_results = []
test_predictions = {}

for name, model in all_eval_models.items():
    p, r, f, pr_auc, scores = evaluate_model(model, test_loader)
    test_predictions[name] = scores
    eval_results.append({
        'Model Pipeline': name,
        'Precision': round(p, 4),
        'Recall': round(r, 4),
        'F1-Score': round(f, 4),
        'PR-AUC': round(pr_auc, 4)
    })

df_eval_results = pd.DataFrame(eval_results)
print("\n=== CROSS-SITE EVALUATION RESULTS ON UNSEEN FACTORY 4 ===")
print(df_eval_results.to_string(index=False))

# Convergence Plot for FL Architectures
plt.figure(figsize=(12, 6))
for key, loss_hist in fl_histories.items():
    if "FedAvg" in key and "DP" not in key:
        plt.plot(range(1, len(loss_hist) + 1), loss_hist, label=key, marker='o')
plt.title('FedAvg Global Convergence Trajectories Across Architectures')
plt.xlabel('Global Round')
plt.ylabel('Average BCE Loss')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

## CELL 7: Knowledge Graph (NetworkX) & Fact Extraction

In [ ]:
print("Constructing Domain Knowledge Graph (FedKG)...")
G = nx.DiGraph()

for model_name in ['model1', 'model2', 'model3', 'model4']:
    G.add_node(model_name, type='MachineModel')

for m_id in df_machines['machineID'].unique()[:20]:
    m_model = df_machines[df_machines['machineID'] == m_id]['model'].values[0]
    m_node = f"Machine_{m_id}"
    G.add_node(m_node, type='Machine')
    G.add_edge(m_node, m_model, relation='Is_Model')

error_comp_mappings = {
    'error1': 'comp1',
    'error2': 'comp2',
    'error3': 'comp3',
    'error4': 'comp4',
    'error5': 'comp1'
}

for err, comp in error_comp_mappings.items():
    G.add_node(err, type='LoggedError')
    G.add_node(comp, type='Component')
    G.add_node(f"Replace_{comp}", type='MaintenanceAction')
    
    G.add_edge(err, comp, relation='Indicates_Wear_In')
    G.add_edge(comp, f"Replace_{comp}", relation='Requires_Action')

def get_kg_facts(machine_id, active_errors, telemetry_anomalies, comp_ages):
    facts = []
    m_node = f"Machine_{machine_id}"
    
    if G.has_node(m_node):
        for neighbor in G.neighbors(m_node):
            facts.append(f"({m_node} -> Is_Model -> {neighbor})")
            
    for err in active_errors:
        if err in error_comp_mappings:
            target_comp = error_comp_mappings[err]
            facts.append(f"({m_node} -> Has_Active_Error -> {err})")
            facts.append(f"({err} -> Indicates_Wear_In -> {target_comp})")
            facts.append(f"({target_comp} -> Requires_Action -> Replace_{target_comp})")
            
    for stat, val in telemetry_anomalies.items():
        facts.append(f"({m_node} -> Has_Telemetry_Anomaly -> {stat}={val:.2f})")
        
    for comp, days in comp_ages.items():
        if days > 30.0:
            facts.append(f"({m_node} -> Component_Age_Exceeded -> {comp}:{days:.1f}_days)")
            facts.append(f"({comp} -> Risk_Level -> HIGH)")
            
    return facts

sample_f4_df = client_data['Factory 4 (Held-Out Test)']['df']
high_risk_sample = sample_f4_df[sample_f4_df['failure_next_24h'] == 1].iloc[0]

sample_m_id = high_risk_sample['machineID']
sample_errors = [e for e in ['error1', 'error2', 'error3', 'error4', 'error5'] if high_risk_sample[f'{e}_count_24h'] > 0]
sample_telemetry = {'volt_mean_24h': high_risk_sample['volt_mean_24h'], 'vibration_std_24h': high_risk_sample['vibration_std_24h']}
sample_comp_ages = {c: high_risk_sample[f'days_since_{c}'] for c in ['comp1', 'comp2', 'comp3', 'comp4']}

kg_triples = get_kg_facts(sample_m_id, sample_errors, sample_telemetry, sample_comp_ages)

print(f"\n=== EXTRACTED KG TRIPLES FOR MACHINE {sample_m_id} ===")
for t in kg_triples:
    print(t)

In [ ]:
## CELL 7.5: Ollama API Connectivity Test
import requests

try:
    res = requests.post(
        "http://localhost:11434/api/generate",
        json={"model": "llama3:latest", "prompt": "Hello", "stream": False},
        timeout=5
    )
    if res.status_code == 200:
        print("Ollama connection successful! Response:", res.json().get("response"))
    else:
        print(f"Ollama returned HTTP error {res.status_code}")
except requests.exceptions.RequestException as e:
    print("Could not connect to Ollama server:", e)

## CELL 8: LLM RAG Grounded Explainability (Ollama Local API)


In [ ]:
def generate_explanation(machine_id, failure_prob, kg_facts=None):
    url = "http://localhost:11434/api/generate"
    
    if kg_facts is None:
        prompt = f"""You are a predictive maintenance assistant.
Machine ID {machine_id} has a predicted failure probability of {failure_prob * 100:.2f}% in the next 24 hours.
Provide diagnostic insights and action recommendations for the maintenance team."""
    else:
        facts_str = "\n".join(kg_facts)
        prompt = f"""You are a predictive maintenance assistant grounded in formal knowledge graphs.
Machine ID: {machine_id}
Predicted Failure Probability: {failure_prob * 100:.2f}%

Retrieved Knowledge Graph Grounding Triples:
{facts_str}

STRICT CONSTRAINTS:
1. Base your diagnosis strictly on the retrieved Knowledge Graph triples provided above.
2. Do not invent or assume unmentioned component failures.
3. Recommend specific replacement actions matching the KG triples directly."""

    payload = {
        "model": "llama3:latest",
        "prompt": prompt,
        "stream": False
    }
    
    try:
        response = requests.post(url, json=payload, timeout=5)
        if response.status_code == 200:
            return response.json().get("response", "No response text received.")
        else:
            return f"[Ollama API Error HTTP {response.status_code}] Grounded Fallback: Inspect components associated with active errors in KG."
    except requests.exceptions.RequestException:
        fallback_text = (
            f"[NETWORK DISCONNECT FALLBACK - OLLAMA OFFLINE]\n"
            f"Machine ID: {machine_id} | Failure Risk: {failure_prob * 100:.1f}%\n"
            f"DETERMINISTIC DIAGNOSTIC RECOMMENDATION:\n"
            f"- High risk detected based on 24-hour window aggregations.\n"
        )
        if kg_facts:
            fallback_text += f"- Grounded KG Action: Execute immediate inspection for facts:\n  " + "\n  ".join(kg_facts)
        return fallback_text

sample_prob = float(test_predictions['FedAvg (LSTM)'][0]) if 'FedAvg (LSTM)' in test_predictions else 0.85

print("=== GENERATING UNGROUNDED LLM EXPLANATION ===")
print(generate_explanation(sample_m_id, sample_prob, kg_facts=None))

print("\n=== GENERATING KG-GROUNDED RAG EXPLANATION ===")
print(generate_explanation(sample_m_id, sample_prob, kg_facts=kg_triples))

## CELL 9: Executive Performance & Diagnostics Matrix

In [ ]:
executive_summary = []

for name in df_eval_results['Model Pipeline']:
    row = df_eval_results[df_eval_results['Model Pipeline'] == name].iloc[0]
    
    if "LSTM" in name:
        arch = "LSTM"
    elif "TCN" in name:
        arch = "TCN"
    elif "1D-CNN" in name:
        arch = "1D-CNN"
    elif "Autoencoder" in name:
        arch = "Autoencoder"
    else:
        arch = "MLP"
        
    if "Centralized" in name:
        paradigm = "Centralized"
        privacy = "None (Data Pooled)"
    elif "FedAvg+DP" in name:
        paradigm = "Federated"
        privacy = "Differential Privacy (ε, δ)"
    elif "FedProx" in name:
        paradigm = "Federated"
        privacy = "Federated Aggregation"
    elif "FedAvg" in name:
        paradigm = "Federated"
        privacy = "Federated Aggregation"
    else:
        paradigm = "Local Only"
        privacy = "Siloed Data"
        
    executive_summary.append({
        'Model Pipeline': name,
        'Architecture': arch,
        'Paradigm': paradigm,
        'Privacy Mechanism': privacy,
        'Precision': row['Precision'],
        'Recall': row['Recall'],
        'F1-Score': row['F1-Score'],
        'PR-AUC': row['PR-AUC']
    })

df_executive_matrix = pd.DataFrame(executive_summary)
print("\n=================================== EXECUTIVE MASTER PERFORMANCE & PRIVACY MATRIX ===================================")
print(df_executive_matrix.to_string(index=False))